## 1. Extracting unlabeled vehicle crops from recording video files

In [9]:
import cv2
import os
from pathlib import Path
from ultralytics import YOLO

In [ ]:
# --- Configuration ---
VIDEO_DIR = Path("E:/heimdall_data")
OUTPUT_DIR = Path("../data/unlabeled_cropsz")
MODEL_PATH = "yolo11s.pt" 
FRAMES_PER_VIDEO = 30
VEHICLE_CLASSES = [2, 3, 5, 7] # COCO: car, motorcycle, bus, truck

In [11]:
print(os.listdir(VIDEO_DIR))
print(os.listdir(r"E:\heimdall_data\SK1 - DW620-20260303T202058Z-1-001\SK1 - DW620"))
print(os.listdir(r"E:\heimdall_data\SK1 - DW620-20260303T202058Z-1-001\SK1 - DW620\GODZINY MIĘDZYSZCZYTOWE 11-13"))

['Arkusze Ruchu-20260303T202054Z-1-001', 'SK1 - DW620-20260303T202058Z-1-001', 'SK1 - DW620-20260303T202058Z-1-002', 'SK10 - DW620-20260303T214656Z-1-001', 'SK10 - DW620-20260303T214656Z-1-002', 'SK12 - DW620-20260303T215257Z-1-001', 'SK12 - DW620-20260303T215257Z-1-002', 'SK14 - DW620-20260303T220633Z-1-001', 'SK15 - DW620-20260303T222408Z-1-001', 'SK15 - DW620-20260303T222408Z-1-002', 'SK16 - DW620-20260303T223030Z-1-001', 'SK16 - DW620-20260303T223030Z-1-002', 'SK17 - DW620-20260303T230454Z-1-001', 'SK17 - DW620-20260303T230454Z-1-002', 'SK18 - DW620-20260303T230908Z-1-001', 'SK18 - DW620-20260303T230908Z-1-002', 'SK18 - DW620-20260303T230908Z-1-003', 'SK19 - DW620-20260303T232559Z-1-001', 'SK19 - DW620-20260303T232559Z-1-002', 'SK2 - DW620-20260303T203054Z-1-001', 'SK2 - DW620-20260303T203054Z-1-002', 'SK20 - DW620-20260303T232600Z-1-001', 'SK20 - DW620-20260303T232600Z-1-002', 'SK21 - DW620-20260303T232602Z-1-001', 'SK21 - DW620-20260303T232602Z-1-002', 'SK3 - DW620-20260303T21235

In [12]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model = YOLO(MODEL_PATH)

In [18]:
def get_video_output_dir(video_path):
    relative_path = video_path.relative_to(VIDEO_DIR)
    return OUTPUT_DIR / relative_path.parent

In [19]:
def process_video(video_path):

    video_output_dir = get_video_output_dir(video_path)
    video_output_dir.mkdir(parents=True, exist_ok=True)

    
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"Failed to open {video_path.name}")
        return

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    interval = max(1, total_frames // FRAMES_PER_VIDEO)
    
    print(f"Processing {video_path.name}: {total_frames} frames, interval {interval}")

    for i in range(FRAMES_PER_VIDEO):
        frame_id = i * interval
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_id)
        ret, frame = cap.read()
        if not ret:
            break

        # Run YOLO inference
        results = model.predict(frame, classes=VEHICLE_CLASSES, conf=0.3, verbose=False)

        for result in results:
            boxes = result.boxes.xyxy.cpu().numpy()
            clss = result.boxes.cls.cpu().numpy()
            
            for idx, (box, cls) in enumerate(zip(boxes, clss)):
                x1, y1, x2, y2 = map(int, box)
                
                # add padding before cropping
                pad = min(10, (x2 - x1) // 5, (y2 - y1) // 5) # 10 pixels or 20% of box size, whichever is smaller

                x1 = max(0, x1 - pad)
                y1 = max(0, y1 - pad)
                x2 = min(frame.shape[1], x2 + pad)
                y2 = min(frame.shape[0], y2 + pad)
                crop = frame[y1:y2, x1:x2]
                
                if crop.size == 0:
                    continue
                
                # Save crop: `video_stem_f{frame_id}_d{idx}.jpg`
                crop_filename = f"{video_path.stem}_f{frame_id}_d{idx}.jpg"
                cv2.imwrite(str(video_output_dir / crop_filename), crop)

    cap.release()

    # Mark video as processed
    (video_output_dir / f"{video_path.stem}_done.txt").touch()


In [20]:
video_files = list(VIDEO_DIR.rglob("*.mp4"))

print(f"Found {len(video_files)} videos.")

for i, v_path in enumerate(video_files):
    video_output_dir = get_video_output_dir(v_path)
    done_flag = video_output_dir / f"{v_path.stem}_done.txt"

    if done_flag.exists():
        print(f"[{i+1}/{len(video_files)}] SKIP {v_path}")
        continue

    print(f"[{i+1}/{len(video_files)}] PROCESS {v_path}")
    process_video(v_path)

print("Extraction Complete!")

Found 2737 videos.
[1/2737] PROCESS E:\heimdall_data\SK1 - DW620-20260303T202058Z-1-001\SK1 - DW620\GODZINY MIĘDZYSZCZYTOWE 11-13\10H59M03S-03600EN-0X00.mp4
Processing 10H59M03S-03600EN-0X00.mp4: 3860 frames, interval 128
[2/2737] PROCESS E:\heimdall_data\SK1 - DW620-20260303T202058Z-1-001\SK1 - DW620\GODZINY MIĘDZYSZCZYTOWE 11-13\11H02M16S-03600EN-0X01.mp4
Processing 11H02M16S-03600EN-0X01.mp4: 3840 frames, interval 128
[3/2737] PROCESS E:\heimdall_data\SK1 - DW620-20260303T202058Z-1-001\SK1 - DW620\GODZINY MIĘDZYSZCZYTOWE 11-13\11H05M28S-03600EN-0X02.mp4
Processing 11H05M28S-03600EN-0X02.mp4: 3840 frames, interval 128
[4/2737] PROCESS E:\heimdall_data\SK1 - DW620-20260303T202058Z-1-001\SK1 - DW620\GODZINY MIĘDZYSZCZYTOWE 11-13\11H08M40S-03600EN-0X03.mp4
Processing 11H08M40S-03600EN-0X03.mp4: 3840 frames, interval 128
[5/2737] PROCESS E:\heimdall_data\SK1 - DW620-20260303T202058Z-1-001\SK1 - DW620\GODZINY MIĘDZYSZCZYTOWE 11-13\11H11M52S-03600EN-0X04.mp4
Processing 11H11M52S-03600EN-0X

## Running data extraction on kaggle dataset

In [ ]:
# import cv2
# import os
# import kagglehub
# from pathlib import Path
# from ultralytics import YOLO

# # 1. Download Dataset
# print("Downloading BIT-Vehicle dataset...")
# os.environ["KAGGLEHUB_CACHE"] = str(Path("../data/kaggle_cache").resolve())

# print(f"Downloading BIT-Vehicle to {os.environ['KAGGLEHUB_CACHE']}...")
# bit_path = Path(kagglehub.dataset_download("kuanghangdong/bitvehicle"))
# image_files = list(bit_path.rglob("*.jpg")) 
# print(f"Found {len(image_files)} images in BIT-Vehicle.")

# # 2. Configuration
# OUTPUT_DIR = Path("../data/unlabeled_crops")
# MODEL_PATH = "yolo11s.pt"
# VEHICLE_CLASSES = [2, 3, 5, 7] # COCO: car, motorcycle, bus, truck
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# model = YOLO(MODEL_PATH)

# # 3. Process Images
# print("Starting extraction from BIT-Vehicle...")
# for i, img_path in enumerate(image_files):
#     # Process every 2nd or 3rd image if you want to limit the size initially
#     if i % 1 != 0: continue 
    
#     frame = cv2.imread(str(img_path))
#     if frame is None: continue

#     results = model.predict(frame, classes=VEHICLE_CLASSES, conf=0.4, verbose=False)

#     for result in results:
#         boxes = result.boxes.xyxy.cpu().numpy()
#         for idx, box in enumerate(boxes):
#             x1, y1, x2, y2 = map(int, box)
            
#             # Add small padding (optional, helps for Class F/Articulated)
#             pad = 5
#             h, w, _ = frame.shape
#             crop = frame[max(0, y1-pad):min(h, y2+pad), max(0, x1-pad):min(w, x2+pad)]
            
#             if crop.size == 0: continue
            
#             # Save using bit prefix to distinguish from video crops
#             crop_filename = f"bit_{img_path.stem}_d{idx}.jpg"
#             cv2.imwrite(str(OUTPUT_DIR / crop_filename), crop)

#     if i % 500 == 0:
#         print(f"Processed {i}/{len(image_files)} images...")

# print("BIT-Vehicle Extraction Complete!")

100%|██████████| 2.50G/2.50G [16:23<00:00, 2.73MB/s]

Extracting files...


Found 9850 images in BIT-Vehicle.
Starting extraction from BIT-Vehicle...
Processed 0/9850 images...
Processed 500/9850 images...
Processed 1000/9850 images...
Processed 1500/9850 images...
Processed 2000/9850 images...
Processed 2500/9850 images...
Processed 3000/9850 images...
Processed 3500/9850 images...
Processed 4000/9850 images...
Processed 4500/9850 images...
Processed 5000/9850 images...
Processed 5500/9850 images...
Processed 6000/9850 images...
Processed 6500/9850 images...
Processed 7000/9850 images...
Processed 7500/9850 images...
Processed 8000/9850 images...
Processed 8500/9850 images...
Processed 9000/9850 images...
Processed 9500/9850 images...
BIT-Vehicle Extraction Complete!
